In [162]:
using WGLMakie
using Bonito
using Printf
using StatsBase
using Graphs
using SimpleWeightedGraphs
using GraphsMatching

# include("../src/main.jl")
# include("../src/mwpm.jl")
include("../src/web_src.jl")

correct (generic function with 1 method)

In [163]:
ρ = [0 0 0 0 1 1 0 0
     0 0 0 0 0 0 0 0
     0 0 0 0 0 0 0 0
     1 1 0 0 0 0 0 1
     1 1 0 0 0 0 0 1
     0 0 0 0 0 0 0 0
     0 0 0 0 0 0 0 0
     0 0 0 0 0 0 0 0]

peff = 0.02
q = 0.01
horizontal_checks, vertical_checks = measure(ρ, q)
horizontal_checks2, vertical_checks2 = heal((horizontal_checks, vertical_checks), peff, q)
domain = track_domains((horizontal_checks2, vertical_checks2))

8×8 Matrix{Bool}:
 0  0  0  0  1  1  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 1  1  0  0  0  0  0  1
 1  1  0  0  0  0  0  1
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0

In [142]:
peff = 0.1
q = 0.01
correct(ρ, peff, q)

8×8 Matrix{Int64}:
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0

In [99]:
function boundary_filter(checks::Tuple)
    horizontal_checks, vertical_checks = deepcopy(checks)
    L = size(horizontal_checks)[1]
    
    horizontal_checks[end,:] .= false
    vertical_checks[:,end] .= false
    
    return horizontal_checks, vertical_checks
end

function boundary_pad(checks::Tuple)
    horizontal_checks, vertical_checks = checks
    L = size(vertical_checks)[1]
    vcs = zeros(eltype(vertical_checks), L+1, L+1)
    vcs[1:L, 2:L] = vertical_checks[:,1:L-1]

    hcs = zeros(eltype(horizontal_checks), L+1, L+1)
    hcs[2:L, 1:L] = horizontal_checks[1:L-1,:]

    return hcs, vcs
end

function measure(ρ::AbstractMatrix, q::Float64)
    horizontal_checks = noiselayer(ρ .⊻ circshift(ρ,(-1,0)),q)
    vertical_checks = noiselayer(ρ .⊻ circshift(ρ,(0,-1)),q)
    checks = horizontal_checks, vertical_checks
    return boundary_pad(boundary_filter(checks))
end

measure (generic function with 1 method)

In [113]:


function detect_charges(hcs::AbstractMatrix, vcs::AbstractMatrix)
    L = size(vcs)[1] - 1

    sites = hcs .⊻ circshift(hcs,(0,1)) .⊻ vcs .⊻ circshift(vcs,(1,0))
    return [(j,i) for i in 0:L, j in 0:L][Bool.(sites)]
end

prob_weight(x::Float64) = -log(x/(1-x))
weight(peff::Float64, q::Float64, m::Bool) = q > 0 ? ((-1)^m * prob_weight(peff) + prob_weight(q)) : (m == true ? 0.0 : Inf)

site(L,x,y) = (L+1)*mod(y,L+1)+mod(x,L+1)+1
unsite(L,s) = (mod1(s,L+1)-1, div(s-1,L+1))

function build_matching_graph(checks::Tuple, peff::Float64, q::Float64)
    horizontal_checks, vertical_checks = checks
    L = size(horizontal_checks)[1]
    
    horizontal_sources = [site(L,x,y) for y in 0:L for x in 0:L-1]
    vertical_sources = [site(L,x,y) for y in 0:L-1 for x in 0:L]
    horizontal_destinations = [site(L,x+1,y) for y in 0:L for x in 0:L-1]
    vertical_destinations = [site(L,x,y+1) for y in 0:L-1 for x in 0:L]
    
    horizontal_weights = zeros(Float64, L+1, L)
    horizontal_weights[2:L, :] = weight.(peff, q, Bool.(horizontal_checks)[1:L-1,:])
    horizontal_weights = reshape(horizontal_weights', L*(L+1), 1)[:,1]

    vertical_weights = zeros(Float64, L, L+1)
    vertical_weights[:,2:L] = weight.(peff, q, Bool.(vertical_checks)[:,1:L-1])
    vertical_weights = reshape(vertical_weights', L*(L+1), 1)[:,1]
    
    return SimpleWeightedGraph([horizontal_sources; vertical_sources], [horizontal_destinations; vertical_destinations], [horizontal_weights; vertical_weights])
end

function match_charges(fw::Graphs.FloydWarshallState, charges::Vector, L::Int)
    subgraph = complete_graph(length(charges))

    weights = Dict{Edge,Float64}()
    for i in 1:length(charges)-1
        for j in i+1:length(charges)
            weights[Edge(i, j)] = fw.dists[site(L,charges[i]...),site(L,charges[j]...)]
        end
    end

    match = minimum_weight_perfect_matching(subgraph, weights)
    return match
end


function heal(checks::Tuple, peff::Float64, q::Float64)
    hcs, vcs = checks
    L = size(hcs)[1]-1

    charges = detect_charges(hcs, vcs)

    if length(charges) == 0
        return hcs, vcs
    end
    
    if peff > q
        g = build_matching_graph((hcs, vcs), peff, q)
    else
        g = build_matching_graph_syndrome_only((hcs, vcs))
    end
    fw = floyd_warshall_shortest_paths(g)
    match = match_charges(fw, charges, L)

    hcs, vcs = apply_paths((hcs, vcs), fw, match, charges, L)

    return hcs, vcs
end

function apply_paths(checks::Tuple, fw::Graphs.FloydWarshallState, match::MatchingResult, charges::Vector, L::Int)
    hcs, vcs = checks
    mated = Int[]
    
    backup_checks = deepcopy(checks)

    for i in 1:length(charges)
        j = match.mate[i]
        if j in mated
            continue
        end
        
        s0 = site(L, charges[i]...)
        s2 = site(L, charges[j]...)
        while s0 != s2
            s1 = fw.parents[s0, s2]
            x1, y1 = unsite(L, s1)
            x2, y2 = unsite(L, s2)
            if x1 == x2
                vcs[max(y1, y2), x1+1] ⊻= true
            elseif y1 == y2
                hcs[y1+1, max(x1, x2)] ⊻= true
            end

            s2 = s1
        end

        push!(mated, i)
        push!(mated, j)
    end
    return hcs, vcs
end

function track_domains(checks::Tuple)
    hcs, vcs = checks
    hcs = Bool.(hcs)
    vcs = Bool.(vcs)
    L = size(hcs)[1]-1

    domain = zeros(Bool, L, L)
    for y in 1:L
        for x in 1:L
            if x == 1 && y == 1
                continue
            elseif x == 1
                domain[y, x] = domain[y-1, x] ⊻ hcs[y, x]
            else
                domain[y, x] = domain[y, x-1] ⊻ vcs[y, x]
            end
        end
    end
    return domain
end

function correct(ρ::AbstractMatrix, peff::Float64, q::Float64)
    checks = measure(ρ, q)
    
    horizontal_checks, vertical_checks = heal(checks, peff, q)
    domain = track_domains((horizontal_checks, vertical_checks))
    if magnetization(domain) > 0.5
        domain = .!domain
    end
    
    return ρ .⊻ domain
end

correct (generic function with 1 method)

In [120]:
ρ = [0 0 0 0 1 1 0 0
     0 0 0 0 0 0 0 0
     0 0 0 0 0 0 0 0
     1 1 0 0 0 0 0 1
     1 1 0 0 0 0 0 1
     0 0 0 0 0 0 0 0
     0 0 0 0 0 0 0 0
     0 0 0 0 0 0 0 0]

horizontal_checks, vertical_checks = measure(ρ, 0.05)
horizontal_checks2, vertical_checks2 = heal((horizontal_checks, vertical_checks), peff, q)
domain = track_domains((horizontal_checks2, vertical_checks2))

8×8 Matrix{Bool}:
 0  0  0  0  1  1  0  0
 0  0  0  0  1  1  1  1
 0  1  1  1  1  1  1  1
 0  0  1  1  1  1  1  0
 0  0  1  1  1  1  1  0
 1  1  1  1  1  1  1  1
 1  1  1  1  1  1  0  1
 1  1  1  1  1  1  1  1

In [118]:
horizontal_checks

9×9 Matrix{Int64}:
 1  1  0  1  0  0  1  1  0
 0  0  0  0  1  1  0  0  0
 0  0  0  0  0  0  0  1  0
 1  1  0  0  0  0  0  1  0
 0  0  0  0  1  1  0  0  0
 0  0  0  0  0  1  1  1  0
 0  0  0  0  0  0  0  0  0
 1  1  1  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0

In [110]:
horizontal_checks == horizontal_checks2

true

In [111]:
vertical_checks == vertical_checks2

true

In [105]:
detect_charges(horizontal_checks, vertical_checks)

6-element Vector{Tuple{Int64, Int64}}:
 (0, 3)
 (0, 5)
 (4, 0)
 (6, 0)
 (8, 3)
 (8, 5)

In [85]:
vertical_checks

8×8 Matrix{Int64}:
 0  0  0  1  0  1  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  1  0  0  0  0  1  0
 0  1  0  0  0  0  1  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0

In [51]:
peff = 0.2
q = 0.1
horizontal_checks, vertical_checks = deepcopy(boundary_filter((horizontal_checks, vertical_checks)))
L = size(horizontal_checks)[1]
    
sources = [site(L,x,y) for y in 0:L for x in 0:L]
horizontal_destinations = [site(L,x+1,y) for y in 0:L for x in 0:L-1]
vertical_destinations = [site(L,x,y+1) for y in 0:L-1 for x in 0:L]

horizontal_weights = zeros(Float64, L+1, L)
horizontal_weights[2:L, :] = weight.(peff, q, Bool.(horizontal_checks)[1:L-1,:])
horizontal_weights = reshape(horizontal_weights', L*(L+1), 1)[:,1]

vertical_weights = zeros(Float64, L, L+1)
vertical_weights[:,2:L] = weight.(peff, q, Bool.(vertical_checks)[:,1:L-1])
vertical_weights = reshape(vertical_weights', L*(L+1), 1)[:,1]

72-element Vector{Float64}:
 0.0
 3.58351893845611
 3.58351893845611
 3.58351893845611
 0.8109302162163285
 3.58351893845611
 0.8109302162163285
 3.58351893845611
 0.0
 0.0
 ⋮
 0.0
 3.58351893845611
 3.58351893845611
 3.58351893845611
 3.58351893845611
 3.58351893845611
 3.58351893845611
 3.58351893845611
 0.0

In [25]:
hcs .⊻ circshift(hcs,(0,1)) .⊻ vcs .⊻ circshift(vcs,(1,0))

9×9 Matrix{Int64}:
 0  0  0  0  1  0  1  0  0
 0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0
 1  0  0  0  0  0  0  0  1
 0  0  0  0  0  0  0  0  0
 1  0  0  0  0  0  0  0  1
 0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0

In [37]:
site(L,x,y) = (L+1)*mod(y,L+1)+mod(x,L+1)+1
unsite(L,s) = (mod1(s,L+1)-1, div(s-1,L+1))

unsite (generic function with 1 method)

In [35]:
site(10,10,10)

121

In [40]:
unsite(10,2)

(1, 0)

In [20]:
padded_ρ = border_pad(ρ)
horizontal_checks, vertical_checks = measure(padded_ρ, 0.0)

detect_charges(horizontal_checks, vertical_checks)

Tuple{Int64, Int64}[]

In [3]:
horizontal_checks = [0 0 0 0 1 1 0 0
                     0 0 0 0 0 0 0 0
                     1 1 0 0 0 0 0 0
                     0 0 0 0 0 0 0 0
                     1 1 0 0 0 0 0 0
                     0 0 0 0 0 0 0 0
                     0 0 0 0 0 0 0 0
                     0 0 0 0 0 0 0 0]

vertical_checks = [0 0 0 1 0 1 0 0
                   0 0 0 0 0 0 0 0
                   0 0 0 0 0 0 0 0
                   0 1 0 0 0 0 0 0 
                   0 1 0 0 0 0 0 0 
                   0 0 0 0 0 0 0 0
                   0 0 0 0 0 0 0 0
                   0 0 0 0 0 0 0 0]


detect_charges(horizontal_checks, vertical_checks)

DimensionMismatch: DimensionMismatch: tried to assign 8×7 array to 9×7 destination

In [27]:
vertical_checks[:,8]

8-element Vector{Int64}:
 1
 0
 0
 0
 0
 0
 0
 0